# Imports

In [44]:
import pandas as pd
import numpy as np

from dateutil.relativedelta import relativedelta

from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

import joblib
import json

np.random.seed(33)

# Load dataset

In [47]:
# LOAD DATA

FILE_PATH = "data/raw/Guadalajara 4Q22.xlsx"

data_ori = pd.read_excel(FILE_PATH, header=0)

data_ori.head()

,ID,Clasificación,Tipo,Nombre,Dirección,Colonia,Municipio,Promotor,Área A,Terraza A,...,Éxito Comercial,Unidades proyectadas,Unidades Totales,Inventario,Vendidas,Fecha de Alta,Fecha de Inicio,Fecha de Actualización,Fecha de Entrega,Financiamiento 1a opción
0,3425,S,CD,Cima Serena Etapa III CD,Av. Del Kinder S/N,El Verde,El Salto,Casas Javer,45.90,0.0,...,0.036794,6000,500,200,300,2022-05-16,2021-07-01,2022-11-09,2022-07-01,INFONAVIT
1,3248,S,CD,El Mirador (CD),Carr. el Salto s/n,El Mirador,El Salto,Casas Bali,60.00,0.0,...,0.017202,5000,412,124,288,2020-08-31,2019-07-01,2022-11-18,2019-11-01,INFONAVIT
2,3195,S,Depto,Valle de Tejeda Javer,Av. Prolongación Colón,Valle de Tejeda,Tlajomulco de Zúñiga,Casas Javer,41.80,0.0,...,0.018048,1200,1200,300,900,2020-02-25,2019-06-01,2022-11-16,2019-08-01,INFONAVIT
3,2839,S,CD,Vistas del Pedregal,Av. Lázaro Cárdenas,Vistas Del Pedregal II,Tonalá,Ruba,53.11,0.0,...,0.009979,4500,2700,110,2590,2017-05-26,2014-11-01,2022-11-03,2017-07-01,INFONAVIT
4,3375,E,CS,Albereda Residencial Coto Encino,Carretera El Salto Km 2,Albereda Residencial,Tlajomulco de Zúñiga,Hogares Unión,65.00,0.0,...,0.004421,5000,250,200,50,2022-02-03,2019-02-01,2022-11-08,2019-09-01,BBVA


# Data Preparation

In [50]:
df = data_ori.copy()

# Reestructura prototipos A/B/C
df_1 = df.iloc[:, 0:8]
df_2 = df.iloc[:, 44:57]
dfA_p = df.iloc[:, 8:20]
dfB_p = df.iloc[:, 20:32]
dfC_p = df.iloc[:, 32:44]

dfA = pd.concat([df_1, dfA_p, df_2], axis=1).rename(columns={
    'Área A':'sqm', 'Terraza A':'terrace', 'Terreno A':'terrain',
    'Recámara A':'bhk', 'Baños A':'baths', 'Alcoba A':'alcoba',
    'Cuarto de Servicio A':'room_serv', 'Cajón A':'park_u',
    'Niveles A':'levels', 'Precio Actual A':'price',
    'Precio Inicial A':'first_price', 'ValorM2 A':'price_per_sqm'
}).dropna()

dfB = pd.concat([df_1, dfB_p, df_2], axis=1).rename(columns={
    'Área B':'sqm', 'Terraza B':'terrace', 'Terreno B':'terrain',
    'Recámara B':'bhk', 'Baños B':'baths', 'Alcoba B':'alcoba',
    'Cuarto de Servicio B':'room_serv', 'Cajón B':'park_u',
    'Niveles B':'levels', 'Precio Actual B':'price',
    'Precio Inicial B':'first_price', 'ValorM2 B':'price_per_sqm'
}).dropna()

dfC = pd.concat([df_1, dfC_p, df_2], axis=1).rename(columns={
    'Área C':'sqm', 'Terraza C':'terrace', 'Terreno C':'terrain',
    'Recámara C':'bhk', 'Baños C':'baths', 'Alcoba C':'alcoba',
    'Cuarto de Servicio C':'room_serv', 'Cajón C':'park_u',
    'Niveles C':'levels', 'Precio Actual C':'price',
    'Precio Inicial C':'first_price', 'ValorM2 C':'price_per_sqm'
}).dropna()

data = pd.concat([dfA, dfB, dfC]).sort_values('ID', ascending=True)

# Renombrado general
data = data.rename(columns={
    'ID':'id',
    'Clasificación':'classification',
    'Tipo':'type',
    'Nombre':'project_name',
    'Dirección':'address',
    'Colonia':'colony',
    'Municipio':'town',
    'Promotor':'promoter',
    'Absorción':'absortion',
    'Meses en venta':'months_in_sale',
    'Meses de inventario':'inventory_months',
    'Éxito Comercial':'comm_succ',
    'Unidades Totales':'total_units',
    'Unidades proyectadas':'master_plan_units',
    'Inventario':'inventory',
    'Vendidas':'sold_units',
    'Fecha de Alta':'entry_date',
    'Fecha de Inicio':'initial_date',
    'Fecha de Actualización':'update_date',
    'Fecha de Entrega':'delivery_date',
    'Financiamiento 1a opción':'fin_op'
}).dropna()

# Limpieza colonia
data['colony'] = data['colony'].str.lower()

colony_patterns = [
    (data['colony'].str.contains('royal co', case=False, regex=False), 'royal country'),
    (data['colony'].str.contains('videncia', case=False, regex=False), 'providencia'),
    (data['colony'].str.contains('ladrón de guevar', case=False, regex=False), 'ladrón de guevara'),
    (data['colony'].str.contains('Villa Bosque (Villa Panamericana)', case=False, regex=False), 'villa bosque'),
    (data['colony'].str.contains('americana', case=False, regex=False), 'americana'),
    (data['colony'].str.contains('lafayette', case=False, regex=False), 'americana'),
    (data['colony'].str.contains('juan ocot', case=False, regex=False), 'san juan de ocotán'),
    (data['colony'].str.contains('santa ana tepet', case=False, regex=False), 'santa ana tepetitlán')
]

criteria, values = zip(*colony_patterns)
data['colony'] = np.select(criteria, values, default=data['colony'])

# Limpieza tipo
data['type'] = data['type'].str.lower()

type_patterns = [
    (data['type'].str.contains('depto', case=False, regex=False), 'depto'),
    (data['type'].str.contains('loft', case=False, regex=False), 'depto'),
    (data['type'].str.contains('cd', case=False, regex=False), 'casa'),
    (data['type'].str.contains('cs', case=False, regex=False), 'casa'),
    (data['type'].str.contains('ch', case=False, regex=False), 'casa'),
    (data['type'].str.contains('town house', case=False, regex=False), 'casa')
]

criteria, values = zip(*type_patterns)
data['type'] = np.select(criteria, values, default=data['type'])

# Variables muertas
data = data.drop([
    'id', 'project_name', 'address', 'promoter',
    'alcoba', 'room_serv', 'first_price',
    'initial_date', 'entry_date', 'update_date',
    'fin_op', 'inventory_months'
], axis='columns', errors='ignore')

# Feature: meses a entrega
base_date = pd.to_datetime('22/09/01', format='%y/%m/%d')
data['delivery_date'] = pd.to_datetime(data['delivery_date'])

def months_diff(fecha):
    rd = relativedelta(fecha, base_date)
    return rd.years * 12 + rd.months + rd.days / 30.44

data['months_to_delivery'] = data['delivery_date'].apply(months_diff)
data.loc[data['months_to_delivery'] < 0, 'months_to_delivery'] = 0
data = data.drop(['delivery_date'], axis='columns')

# Clasificación a numérico
clas_soft = ['S', 'E', 'M', 'R', 'RP']
mapeo = {k: i for i, k in enumerate(clas_soft, -2)}
data['classification'] = data['classification'].map(mapeo)

# Filtros finales
data = data[data['type'] == 'depto'].copy()
data = data.drop(['terrain', 'type'], axis='columns', errors='ignore')
data = data.drop(data[data['town'] == 'Chapala'].index)
data = data.drop(data[data['town'] == 'El Salto'].index)
data = data[(data.price_per_sqm < 90000) & (data.sqm < 250.0)]

# Dummies y dataset final
dummies = pd.get_dummies(data['town'])
data = pd.concat([data, dummies], axis='columns')

data = data.drop([
    'colony', 'town', 'price_per_sqm',
    'comm_succ', 'absortion', 'sold_units', 'baths'
], axis='columns', errors='ignore')

data.head()

,classification,sqm,terrace,bhk,park_u,levels,price,months_in_sale,master_plan_units,total_units,inventory,months_to_delivery,Guadalajara,Jocotepec,Tlajomulco de Zúñiga,Tlaquepaque,Tonalá,Zapopan
195,1,222.0,0.0,3.0,2.0,11.0,9077955.0,85.150685,81,44,4,0.0,True,False,False,False,False,False
195,1,111.0,0.0,2.0,2.0,11.0,5922418.0,85.150685,81,44,4,0.0,True,False,False,False,False,False
195,1,110.0,0.0,2.0,2.0,11.0,5867088.0,85.150685,81,44,4,0.0,True,False,False,False,False,False
226,1,91.0,0.0,2.0,2.0,10.0,5300000.0,84.295890,392,392,8,0.0,False,False,False,False,False,True
226,1,131.0,0.0,3.0,2.0,10.0,6503000.0,84.295890,392,392,8,0.0,False,False,False,False,False,True


# Split dataset

In [53]:
X_df = data.drop(['price'], axis='columns')
y_df = data['price']

feature_names = X_df.columns.tolist()

X = X_df.values
Y = y_df.values.reshape(-1, 1)

xtrain, xtest, ytrain, ytest = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

scaler_X = StandardScaler()
scaler_Y = StandardScaler()

xtrain = scaler_X.fit_transform(xtrain)
xtest = scaler_X.transform(xtest)

ytrain = scaler_Y.fit_transform(ytrain).ravel()
ytest = scaler_Y.transform(ytest).ravel()

xtrain.shape, xtest.shape

((428, 17), (108, 17))

# Train

In [56]:
alphas = [1000, 500, 200, 100, 50, 20, 10, 1, 0.1, 0.01]
folds = KFold(n_splits=10, shuffle=True, random_state=42)

ridge_grid = GridSearchCV(
    estimator=Ridge(),
    param_grid={'alpha': alphas},
    scoring='r2',
    cv=folds,
    n_jobs=-1
)

ridge_grid.fit(xtrain, ytrain)

modelo_final = ridge_grid.best_estimator_

print("Mejores hiperparámetros:", ridge_grid.best_params_)
print("R2 CV train:", ridge_grid.best_score_)

Mejores hiperparámetros: {'alpha': 20}
R2 CV train: 0.8533903044819209


# Evaluate

In [59]:
y_pred_train = modelo_final.predict(xtrain)
y_pred_test = modelo_final.predict(xtest)

r2_train = r2_score(ytrain, y_pred_train)
r2_test = r2_score(ytest, y_pred_test)

ytrain_real = scaler_Y.inverse_transform(ytrain.reshape(-1, 1)).ravel()
ytest_real = scaler_Y.inverse_transform(ytest.reshape(-1, 1)).ravel()

y_pred_train_real = scaler_Y.inverse_transform(y_pred_train.reshape(-1, 1)).ravel()
y_pred_test_real = scaler_Y.inverse_transform(y_pred_test.reshape(-1, 1)).ravel()

metrics = {
    "R2_train": r2_train,
    "R2_test": r2_test,
    "RMSE_train_scaled": root_mean_squared_error(ytrain, y_pred_train),
    "RMSE_test_scaled": root_mean_squared_error(ytest, y_pred_test),
    "MAE_train_scaled": mean_absolute_error(ytrain, y_pred_train),
    "MAE_test_scaled": mean_absolute_error(ytest, y_pred_test),
    "RMSE_train_pesos": root_mean_squared_error(ytrain_real, y_pred_train_real),
    "RMSE_test_pesos": root_mean_squared_error(ytest_real, y_pred_test_real),
    "MAE_train_pesos": mean_absolute_error(ytrain_real, y_pred_train_real),
    "MAE_test_pesos": mean_absolute_error(ytest_real, y_pred_test_real)
}

joblib.dump(modelo_final, "modelo_final.pkl")
joblib.dump(scaler_X, "scaler_X.pkl")
joblib.dump(scaler_Y, "scaler_Y.pkl")

with open("feature_names.json", "w", encoding="utf-8") as f:
    json.dump(feature_names, f, ensure_ascii=False, indent=4)

with open("metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=4)

metrics

{'R2_train': 0.8722779520936733,
 'R2_test': 0.8586681107871221,
 'RMSE_train_scaled': 0.3573822154309398,
 'RMSE_test_scaled': 0.3153479290412656,
 'MAE_train_scaled': 0.2606522835231308,
 'MAE_test_scaled': 0.23925633426892443,
 'RMSE_train_pesos': 820764.760797348,
 'RMSE_test_pesos': 724228.7287166608,
 'MAE_train_pesos': 598614.5921648006,
 'MAE_test_pesos': 549476.6093178215}